# 01 · Failure demo for Q&A 1
**Advanced GenAI HS26 · Trustworthy Agentic RAG · Group 2: Divya Shori, Faranak Rahimi, Hanieh Jebeli**

Shows five failures of the reference system **from its real decision traces**, with three views for each:
- **Replay:** the reference team's saved trace. It's instant, costs nothing, and gives the same result every time.
- **Index check:** a live search of the indexed chunks (`docs_fixed_norm.pkl`) to show what the system *could* have found.
- **Live (optional):** the same question run on the system now (about 10–60 s, a few cents). The result can differ from the replay; that is example 4.

**Before the call:** Runtime → Run all (about 10 min for the first model download). Section 4 loads the live system; set `LIVE = False` to skip it and use replay only.

## 1 · Setup (same as notebook 00)

In [ ]:
# ---- edit if your Drive layout differs --------------------------------------
WORK_DIR_NAME   = "AdvGenAI"                 # our group folder in My Drive (outputs go here)
ARTIFACT_FOLDER = "RAG_Project_Artifact"     # name of the shortcut to the shared folder
COPY_TO_LOCAL   = True    # copy indexes to Colab disk: faster, and avoids Chroma writing to a read-only folder

# Models: the reference run used these. Keep them for reproduction; a cheaper
# generator model is fine for exploratory runs, but note it in the log.
AGENT_MODEL_GENERATION = "claude-opus-4-7"
AGENT_MODEL_VERIFIER   = "claude-opus-4-7"
AGENT_MODEL_UTILITY    = "claude-haiku-4-5"
LIVE = True    # False = replay + index checks only (no model download, no API cost)


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, pathlib, platform, datetime
ISSUES = []
def log_issue(step, err=None, note=""):
    tb = traceback.format_exc() if err is not None else ""
    ISSUES.append({"time": datetime.datetime.now().isoformat(timespec="seconds"),
                   "step": step, "error": repr(err) if err else "", "note": note, "traceback": tb[-3000:]})
    print(f"⚠️  logged issue at [{step}]: {err or note}")

def step(name):
    # Use as: with step("x"): ...  → exceptions are logged, not raised
    class _S:
        def __enter__(self): print(f"\n▶ {name}"); self.t = time.time()
        def __exit__(self, et, ev, tb):
            if ev is not None:
                log_issue(name, ev); return True
            print(f"  ✓ {name} ({time.time()-self.t:.0f}s)")
    return _S()

print("Python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    if not torch.cuda.is_available():
        log_issue("gpu", note="No GPU runtime: model loading/retrieval will be very slow on CPU.")
except Exception as e:
    log_issue("torch import", e)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = pathlib.Path("/content/drive/MyDrive")
WORK = MYDRIVE / WORK_DIR_NAME
WORK.mkdir(parents=True, exist_ok=True)

def find_one(pattern, roots, maxdepth=4):
    for root in roots:
        for depth in range(maxdepth + 1):
            hits = list(root.glob("/".join(["*"] * depth + [pattern])) if depth else root.glob(pattern))
            if hits: return hits[0]
    return None

with step("locate shared full_corpus"):
    FULL_CORPUS_SRC = None
    cand = MYDRIVE / ARTIFACT_FOLDER / "storage" / "full_corpus"
    FULL_CORPUS_SRC = cand if cand.is_dir() else find_one("storage/full_corpus", [MYDRIVE], 3)
    assert FULL_CORPUS_SRC, ("full_corpus not found. Did you add the RAG_Project_Artifact shortcut to My Drive?")
    print("  full_corpus:", FULL_CORPUS_SRC)
    for sub in ["retrieval/fixed_size_chunk/bm25_retriever_full.pkl", "vectordb_dense/fixed_e5/chroma.sqlite3",
                "retrieval_graph/comm2chunk_fixed.json", "retrieval_graph/embeddings", "Lang_norm/fixed_size_chunk"]:
        p = FULL_CORPUS_SRC / sub
        print(f"   {'OK ' if p.exists() else '!! '} {sub}")
        if not p.exists(): log_issue("artifact check", note=f"missing in shared folder: {sub}")

with step("locate Baseline.zip"):
    ZIP = WORK / "Baseline.zip"
    ZIP = ZIP if ZIP.exists() else find_one("Baseline.zip", [MYDRIVE], 3)
    print("  Baseline.zip:", ZIP)

In [ ]:
REPO = pathlib.Path("/content/Baseline")          # repo root (contains src/, results/, Baseline/benchmark)
with step("unpack reference code"):
    if not (REPO / "src").exists():
        if ZIP:
            subprocess.run(["unzip", "-q", "-o", str(ZIP), "-d", "/content"], check=True)
        else:
            log_issue("code source", note="Baseline.zip not found, so cloning GitHub (no qrels: retrieval metrics unavailable)")
            subprocess.run(["git", "clone", "-q", "https://github.com/CyrielVanHelleputte/Adv.-Gen-AI", str(REPO)], check=True)
    assert (REPO / "src" / "agentic" / "pipeline.py").exists(), "unexpected repo layout"
    print("  repo:", REPO)

BASELINE_ROOT = REPO / "Baseline"                 # what config.py expects
STORAGE_LINK  = BASELINE_ROOT / "storage" / "full_corpus"
with step("stage full_corpus"):
    STORAGE_LINK.parent.mkdir(parents=True, exist_ok=True)
    if STORAGE_LINK.is_symlink() or STORAGE_LINK.exists():
        print("  already staged:", STORAGE_LINK.resolve())
    elif COPY_TO_LOCAL:
        local = pathlib.Path("/content/full_corpus")
        if not local.exists():
            print("  copying shared indexes to local disk (a few minutes)...")
            subprocess.run(["rsync", "-a", "--exclude", "__pycache__", f"{FULL_CORPUS_SRC}/", f"{local}/"], check=True)
        STORAGE_LINK.symlink_to(local)
    else:
        STORAGE_LINK.symlink_to(FULL_CORPUS_SRC)
    print("  du:", subprocess.run(["du", "-sh", str(STORAGE_LINK.resolve())], capture_output=True, text=True).stdout.strip())

In [ ]:
%%capture pip_out
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-core>=0.3,<0.4" \
    "langchain-huggingface>=0.1,<0.4" chromadb "rank_bm25>=0.2.2" "sentence-transformers>=3.0" \
    "pytrec_eval-terrier>=0.5.7" langdetect sentencepiece "anthropic>=0.69" python-dotenv

In [ ]:
tail = pip_out.stdout[-1500:] + pip_out.stderr[-1500:]
if "ERROR" in tail: log_issue("pip install", note=tail)
from importlib.metadata import version
VERSIONS = {p: version(p) for p in ["langchain","langchain-community","langchain-core","langchain-huggingface",
            "chromadb","sentence-transformers","transformers","torch","anthropic","numpy"]}
print(json.dumps(VERSIONS, indent=1))

In [ ]:
from google.colab import userdata
with step("read ANTHROPIC_API_KEY secret"):
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["BASELINE_ROOT"] = str(BASELINE_ROOT)
os.environ["AGENT_MODEL_GENERATION"] = AGENT_MODEL_GENERATION
os.environ["AGENT_MODEL_VERIFIER"]   = AGENT_MODEL_VERIFIER
os.environ["AGENT_MODEL_UTILITY"]    = AGENT_MODEL_UTILITY

os.chdir(REPO)
SRC = str(REPO / "src")
if SRC not in sys.path: sys.path.insert(0, SRC)


## 2 · Benchmark questions + demo helpers

In [ ]:
qa = json.loads((BASELINE_ROOT / "benchmark" / "benchmark_qa_bilingual.json").read_text(encoding="utf-8"))
QA = {str(x["id"]): x for x in qa}
rag = None
print(len(QA), "benchmark questions")


In [ ]:
# ---- Demo helpers: replay a trace, search the index, run live ----------------
import json, html, re, pickle, glob, time, pathlib
from IPython.display import display, HTML

TRACES_REF = REPO / "results" / "agentic" / "traces"            # reference team's run
LABELS     = json.loads((BASELINE_ROOT / "benchmark" / "reliability_labels.json").read_text(encoding="utf-8"))
OUR_RUNS   = sorted(glob.glob(str(WORK / "runs" / "*" / "benchmark" / "**" / "*.json"), recursive=True))

COLORS = {"answer": "#2F6B4F", "abstain": "#B23A1E", "clarify": "#9A6700"}
CSS = ("<style>.tr{font-family:Arial,sans-serif;font-size:15px;border-collapse:collapse;width:100%;margin:6px 0}"
       ".tr td,.tr th{border-bottom:1px solid #ddd;padding:6px 8px;text-align:left;vertical-align:top}"
       ".tr th{background:#14213D;color:#fff}.k{font-weight:bold;white-space:nowrap}"
       ".q{font-family:Arial,sans-serif;font-size:20px;font-weight:bold;margin:14px 0 4px}"
       ".fin{font-family:Arial,sans-serif;font-size:18px;padding:10px 12px;border-radius:8px;color:#fff;margin:6px 0}"
       ".snip{font-family:Arial,sans-serif;font-size:15px;margin:4px 0 12px;padding:8px 10px;background:#FFFDF3;border-left:4px solid #C2571A}"
       "mark{background:#FFD27A}</style>")

def _e(x): return html.escape(str(x))

def show(t, title=""):
    """Render one decision trace as a readable table."""
    p, f = t.get("profile", {}), t["final"]
    rows = []
    for s in t["timeline"]:
        k, sec = s["step"], s.get("t_ms", 0) / 1000
        if   k == "profile":  d = f"type={p.get('query_type')} · ambiguous={p.get('ambiguity_flag')}" + (f" · <i>{_e(p.get('ambiguity_reason'))}</i>" if p.get("ambiguity_reason") else "") + f" · start mode={s.get('initial_mode')}"
        elif k == "retrieve": d = f"<b>{s['mode']}</b> · query: <i>{_e(s['query'][:110])}</i>"
        elif k == "analyze":  d = " · ".join(f"{a}={b}" for a, b in s["signals"].items() if a in ("sufficiency", "chunk_agreement", "grounding", "confidence"))
        elif k == "generate": d = f"evidence_found=<b>{s.get('evidence_found')}</b> · <i>“{_e(s.get('draft_answer', '')[:260])}”</i>"
        elif k == "verify":   d = f"grounded=<b>{s.get('grounded')}</b> · score={s.get('grounding_score')} · <i>{_e(s.get('reason', '')[:200])}</i>"
        elif k == "recover":  d = f"{s.get('old_mode')} → <b>{s.get('new_mode')}</b> · rewritten: <i>{_e(s.get('rewritten_query', '')[:110])}</i>"
        elif k == "decide":   d = f"→ <b>{s['action']}</b> · {_e(s['reason'])}"
        elif k == "final":    continue
        else:                 d = _e({a: b for a, b in s.items() if a not in ("step", "t_ms")})[:200]
        rows.append(f"<tr><td class=k>{k.upper()}</td><td>{sec:.1f}</td><td>{d}</td></tr>")
    a = f.get("action", "?")
    txt = f.get("answer") or f.get("clarifying_question") or f.get("abstain_reason") or ""
    display(HTML(CSS + f"<div class=q>{_e(title)} [{t['qid']}] {_e(t['query'])}</div>"
                 "<table class=tr><tr><th>Step</th><th>t (s)</th><th>What happened</th></tr>" + "".join(rows) + "</table>"
                 f"<div class=fin style='background:{COLORS.get(a, '#555')}'>FINAL: <b>{a.upper()}</b> · confidence {f.get('confidence')}"
                 f"<br><span style='font-size:15px'>{_e(txt[:600])}</span></div>"))

def gold(qid):
    """What the benchmark expects for this question."""
    L = LABELS[str(qid)]
    display(HTML(CSS + f"<div class=snip><b>Benchmark label:</b> expected <b>{L['expected_behavior']}</b> · "
                 f"{L.get('gold_chunks') if isinstance(L.get('gold_chunks'), int) else len(L.get('gold_chunks') or [])} gold chunks<br><b>Gold answer:</b> {_e(L.get('gold_answer'))}</div>"))

def replay(qid):
    """Reference team's saved trace: instant, no API cost."""
    show(json.loads((TRACES_REF / f"{qid}.json").read_text(encoding="utf-8")), "Reference run")

def ours(qid):
    """Our 17 Nov Colab run of the same question (if saved on Drive)."""
    hits = [p for p in OUR_RUNS if pathlib.Path(p).name == f"{qid}.json"]
    if not hits: return print("No saved trace from our run for", qid)
    show(json.loads(pathlib.Path(hits[-1]).read_text(encoding="utf-8")), "Our run")

def live(qid_or_question):
    """Run the question on the live system now (≈10–60 s, a few cents)."""
    assert rag is not None, "Load the live system first (section 4)."
    qid = str(qid_or_question)
    item = QA.get(qid)
    q = item["question"] if item else qid
    t0 = time.perf_counter()
    t = rag.answer(qid if item else "live", q, item).to_dict()
    show(t, f"LIVE now ({time.perf_counter() - t0:.0f} s)")
    return t

_CHUNKS = None
def _load_chunks():
    global _CHUNKS
    if _CHUNKS is None:
        docs = pickle.load(open(STORAGE_LINK / "Lang_norm" / "fixed_size_chunk" / "docs_fixed_norm.pkl", "rb"))
        md = lambda d: getattr(d, "metadata", None) or d.get("metadata", {})
        _CHUNKS = [(md(d).get("chunk_id", ""), md(d).get("filename", ""), md(d).get("original_text", "")) for d in docs]
        print(f"{len(_CHUNKS):,} indexed chunks loaded")
    return _CHUNKS

def _norm(s):
    s = s.lower()
    for a, b in (("ä", "ae"), ("ö", "oe"), ("ü", "ue"), ("ß", "ss")): s = s.replace(a, b)
    return s

def evidence(*terms, n=5):
    """Search the indexed chunks (what the system can retrieve) for chunks containing ALL terms."""
    T = [_norm(x) for x in terms]
    hits = [(c, f, t) for c, f, t in _load_chunks() if all(x in _norm(t) for x in T)]
    out = [CSS + f"<div class=q>Index search {_e(' + '.join(terms))}: {len(hits)} chunk(s)</div>"]
    for c, f, t in hits[:n]:
        nt = _norm(t); i = nt.find(T[0]); s = t[max(0, i - 220): i + 260]
        s = _e(s)
        for x in terms: s = re.sub("(" + re.escape(_e(x)) + ")", r"<mark>\1</mark>", s, flags=re.I)
        out.append(f"<div class=snip><b>{_e(c)}</b> · {_e(f)}<br>…{s}…</div>")
    if not hits: out.append("<div class=snip>No indexed chunk contains this. The system cannot find it, whatever it does.</div>")
    display(HTML("".join(out)))


In [ ]:
evidence("Olaf Kübler")   # loads the index once (about 20 s) and checks that search works


## 3 · Load the live system (optional)
The first run downloads the models; expect 5–10 minutes. Skip this if `LIVE = False`.

In [ ]:
from baseline import config
from agentic import AgenticRAG
if LIVE:
  with step("AgenticRAG.from_pretrained()"):
      print("  device:", config.DEVICE, "| BASELINE_ROOT:", config.BASELINE_ROOT)
      print("  models:", config.AGENT_MODEL_GENERATION, config.AGENT_MODEL_VERIFIER, config.AGENT_MODEL_UTILITY)
      rag = AgenticRAG.from_pretrained()

---
## Example 1 · Q1 “Who was president of ETH in 2003?”: a false abstention
**Stage:** retrieval + benchmark label.
**Point to show:** the system tries three retrieval modes and abstains, yet the answer (Olaf Kübler) is in an indexed chunk. Read the generator lines: it finds Ursprung and Eichler, but never Kübler. Sufficiency says “enough” every time (0.60–0.83).
Note the label too: it says *abstain*, so the reference evaluation scored this as correct.

In [ ]:
replay("1")
gold("1")


In [ ]:
evidence("Olaf Kübler")      # the answer IS in the index


In [ ]:
# live("1")   # optional: run it now


---
## Example 2 · Q20 student-number projections: honest text, wrong decision
**Stage:** decision policy + benchmark.
**Point to show:** the generator first says *evidence_found=False*, then *True* for the same message (“do not give specific numerical projections”). The verifier passes it (grounding 1.00) and the policy files it as an **ANSWER at confidence 0.91**.
Then the index check shows that the gold fact (3.5% per year) is **in no chunk**. The text was true, so the right action was to abstain, and the benchmark gold is unreachable. Note that the label lists 57 “gold chunks”: relevant chunks, not chunks that hold the answer.

In [ ]:
replay("20")
gold("20")


In [ ]:
evidence("3.5%", "student")
evidence("3.5 per cent")


---
## Example 3 · Confidence doesn't separate right from wrong
**Stage:** decision policy.
**Point to show:** correct, partially correct and incorrect answers all get about 0.9 confidence (ECE 0.47). Confidence is computed after the action is chosen, so no decision uses it.

In [ ]:
import matplotlib.pyplot as plt
M = json.loads((REPO / "results" / "agentic" / "evaluation_metrics.json").read_text())["correctness"]
cal = json.loads((REPO / "results" / "agentic" / "calibration.json").read_text())
order = ["correct", "partially_correct", "incorrect"]
pts = {v: [x["confidence"] for x in M["per_query"].values() if x["verdict"] == v] for v in order}
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, v in enumerate(order):
    ax.scatter(pts[v], [i] * len(pts[v]), s=90, alpha=.7, label=f"{v} (n={len(pts[v])}, mean {sum(pts[v])/len(pts[v]):.2f})")
ax.set_yticks(range(3)); ax.set_yticklabels(order); ax.set_xlim(0.7, 1.0); ax.set_xlabel("confidence")
ax.set_title(f"Reference run: confidence by judged correctness  ·  ECE {cal['ece']:.2f}"); ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()


---
## Example 4 · Q5 “What did Prof. Schubert say about flying?”: clarify before retry, and unstable
**Stage:** decision policy.
**Point to show:** the profiler flags ambiguity (“Multiple professors with surname Schubert may exist”). That's true, as the index search shows, but only Renate Schubert wrote about flying. The policy clarifies after one weak BM25 search instead of trying another mode. In our rerun the same question was answered correctly, and 4 of 25 actions flipped between runs. A live run here may give either result.

In [ ]:
replay("5")
gold("5")


In [ ]:
evidence("schubert", "flying")


In [ ]:
ours("5")        # our 17 Nov run, if its traces are on Drive
# live("5")      # optional: see which way it goes today


---
## Example 5 · Q18 “What connections does ETH have to Hong Kong?”: evidence exists, yet it abstains
**Stage:** benchmark + retrieval.
**Point to show:** the stored query has a PDF typo (“conneczons”). The rewrite fixes it and still misses. The index has a few Hong Kong chunks, some only in German (“Hongkong”). The gold answer (CUHK/HKUST agreements) is in no chunk. The right action is a partial answer.

In [ ]:
replay("18")
gold("18")


In [ ]:
evidence("hong kong", n=4)
evidence("hongkong", n=2)
evidence("CUHK")


---
## Try any question live
Type a question from the audience, or a benchmark id such as `"12"`.

In [ ]:
# live("Who was president of ETH in 2003?")
